### 多工具并行调用

In [25]:
import json
import os

import dotenv
import httpx

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 加载环境变量
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


@tool
def get_weather(loc: str) -> str:
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"
    # Step 2.设置查询参数
    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }
    # Step 3.发送GET请求
    response = httpx.get(url, params=params)
    # Step 4.解析响应
    data = response.json()

    return json.dumps(data, ensure_ascii=False)


# 初始化模型（新版推荐）
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=deepseek_api_key
)

# 创建 Agent（最新版推荐）
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="你是天气助手，请根据用户的问题，给出相应的天气信息。"
)

# 调用 Agent
for step in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "请问今天北京和上海天气怎么样，哪个城市更热？"}
        ]
    }
):
    print("\n================ STEP ================")
    print(step)


================ STEP ================
{'model': {'messages': [AIMessage(content='好的，我先查询一下北京和上海的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 400, 'total_tokens': 487, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '72179733-b8f6-4c2e-9bd6-0398f579193c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e248f-364e-7230-bae1-3753525d67d0-0', tool_calls=[{'name': 'get_weather', 'args': {'loc': 'Beijing'}, 'id': 'call_00_kiLPkCcIubjRAYEzGVSt8175', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'loc': 'Shanghai'}, 'id': 'call_01_ofPl5uqHtAhIULJJf1u07676', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

### 多工具串联调用

In [36]:
import json
import os
import dotenv
import httpx

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

from langgraph.prebuilt import create_react_agent


# =========================
# 加载环境变量
# =========================
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


# =========================
# 工具1：天气查询
# =========================
@tool
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 城市英文名（如 Beijing / Shanghai）
    :return: 天气JSON字符串
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }

    response = httpx.get(url, params=params)
    data = response.json()

    return json.dumps(data, ensure_ascii=False)


# =========================
# 工具2：写文件
# =========================
@tool
def write_file(content):
    """
    将指定内容写入本地文件
    """

    print(f"写入文件内容：{content}")

    with open("result.txt", "w", encoding="utf-8") as f:
        f.write(content)

    return "已成功写入本地文件。"


# =========================
# 初始化 LLM（DeepSeek在线）
# =========================
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=deepseek_api_key
)


# =========================
# 创建工具列表
# =========================
tools = [get_weather, write_file]


# =========================
# 创建 Agent（关键替换点）
# =========================
agent = create_react_agent(
    model=llm,
    tools=tools,
)


# =========================
# 执行任务
# =========================
result = agent.invoke(
    {
        "messages": [
            ("user", "查一下北京和上海现在的温度，并将结果写入本地的文件中。")
        ]
    }
)


# =========================
# 打印结果（最终答案）
# =========================
print("\n========== 最终结果 ==========\n")
print(result["messages"][-1].content)

C:\Users\zhanghailong\AppData\Local\Temp\ipykernel_9956\4240316208.py:83: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


写入文件内容：北京和上海当前温度查询结果
查询时间：根据数据时间戳换算

【北京】🌤
温度：27.94°C
体感温度：27.48°C
天气：阴，多云
湿度：38%
风速：2.19 m/s

【上海】☁️
温度：27.92°C
体感温度：29.02°C
天气：多云
湿度：57%
风速：6.0 m/s


========== 最终结果 ==========

已完成！以下是查询结果的总结：

---

### 📍 北京
- **温度**：**27.94°C**
- **体感温度**：27.48°C
- **天气**：阴，多云
- **湿度**：38%

### 📍 上海
- **温度**：**27.92°C**
- **体感温度**：29.02°C
- **天气**：多云
- **湿度**：57%

可以看到，北京和上海当前温度非常接近（相差仅0.02°C），但北京的湿度较低，体感更干爽；上海湿度较高，体感温度略高于实际气温。

结果已成功写入本地文件中 📄
